# 作业 4
**姓名：** 丁平尖  
**日期：** 2026年6月17日

---

## 注意事项
1. 编程题需要打印相应的输出；
2. 将ipynb文件提交至github上，命名为：HW04-学号-姓名.ipynb

---

## 2 序列模型

### 2.1 理论计算题

**题目：** 给定一个字符序列"ababc"，假设采用一阶马尔可夫模型（即 $p(x_t | x_{t-1})$），使用拉普拉斯平滑（加1平滑）估计以下条件概率。词汇表为 {'a','b','c'}，计算时考虑所有可能转移，包括未出现的情况。

**分析序列 "ababc"：**

字符转移对为：
- a → b
- b → a
- a → b
- b → c

**统计转移次数：**

| 转移 | 次数 |
|------|------|
| a → a | 0 |
| a → b | 2 |
| a → c | 0 |
| b → a | 1 |
| b → b | 0 |
| b → c | 1 |
| c → a | 0 |
| c → b | 0 |
| c → c | 0 |

**拉普拉斯平滑公式：**

$$p(x_t | x_{t-1}) = \frac{count(x_{t-1} \rightarrow x_t) + 1}{count(x_{t-1}) + |V|}$$

其中 $|V| = 3$（词汇表大小）

**统计前缀出现次数：**

- count('a') = 2（作为前缀出现2次）
- count('b') = 2（作为前缀出现2次）
- count('c') = 0（作为前缀出现0次）

**1. 计算 $p('a' | 'b')$：**

$$p('a' | 'b') = \frac{count(b \rightarrow a) + 1}{count(b) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = \mathbf{0.4}$$

**2. 计算 $p('c' | 'b')$：**

$$p('c' | 'b') = \frac{count(b \rightarrow c) + 1}{count(b) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = \mathbf{0.4}$$

### 2.2 编程题 - 文本预处理函数

要求：编写一个函数 `preprocess_text(text, n)`，完成以下步骤：
1. 将文本转换为小写，去除标点符号（保留字母和空格）
2. 按空格分词
3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签（用于自回归语言模型）

返回词汇表字典和(特征列表, 标签列表)。

In [ ]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数
    
    参数:
        text: 输入文本字符串
        n: 滑动窗口大小（特征序列长度）
    
    返回:
        vocab: 词汇表字典 {word: id}
        (features, labels): 特征列表和标签列表
    """
    # 1. 将文本转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    # 按频率降序排序，频率相同则按字母顺序
    sorted_words = sorted(word_counts.keys(), key=lambda w: (-word_counts[w], w))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(words) - n):
        # 特征：从位置i开始的n个词
        feature = words[i:i + n]
        # 标签：特征序列后的下一个词
        label = words[i + n]
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)


# 测试代码
print("=" * 50)
print("文本预处理测试")
print("=" * 50)

# 测试1: 题目示例
text1 = "The time machine"
n1 = 2
vocab1, (features1, labels1) = preprocess_text(text1, n1)

print(f"\n输入文本: {text1}")
print(f"窗口大小 n: {n1}")
print("\n词汇表 (按频率排序):")
for word, idx in vocab1.items():
    print(f"  {word}: {idx}")
print(f"\n特征序列: {features1}")
print(f"标签序列: {labels1}")

# 测试2: 更复杂的文本
print("\n" + "=" * 50)
print("更复杂文本测试")
print("=" * 50)

text2 = "Hello world hello python"
n2 = 2
vocab2, (features2, labels2) = preprocess_text(text2, n2)

print(f"\n输入文本: {text2}")
print(f"窗口大小 n: {n2}")
print("\n词汇表 (按频率排序):")
for word, idx in vocab2.items():
    print(f"  {word}: {idx}")
print(f"\n特征序列: {features2}")
print(f"标签序列: {labels2}")

文本预处理测试

输入文本: The time machine
窗口大小 n: 2

词汇表 (按频率排序):
  the: 0
  time: 1
  machine: 2

特征序列: [['the', 'time'], ['time', 'machine']]
标签序列: ['machine']

更复杂文本测试

输入文本: Hello world hello python
窗口大小 n: 2

词汇表 (按频率排序):
  hello: 0
  world: 1
  python: 2

特征序列: [['hello', 'world'], ['world', 'hello'], ['hello', 'python']]
标签序列: ['hello', 'python']


---

## 3 循环神经网络

### 3.1 理论计算题

**题目：** 考虑一个线性RNN（无偏置），定义为 $h_t = W_{hh} h_{t-1} + W_{hx} x_t$，输出 $o_t = W_{oh} h_t$。假设损失函数为平方损失 $L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$。推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

**推导过程：**

**步骤1：定义前向传播**

$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$
$$o_t = W_{oh} h_t$$
$$L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$$

**步骤2：计算损失对 $h_t$ 的梯度**

首先，损失对 $h_t$ 的梯度包含两部分：
- 直接来自当前时间步输出 $o_t$ 的梯度
- 来自未来时间步通过 $h_t$ 传递的梯度

$$\frac{\partial L}{\partial h_t} = W_{oh}^T (o_t - y_t) + W_{hh}^T \frac{\partial L}{\partial h_{t+1}}$$

定义 $\delta_t = \frac{\partial L}{\partial h_t}$，则：

$$\delta_t = W_{oh}^T (o_t - y_t) + W_{hh}^T \delta_{t+1}$$

边界条件：$\delta_{T+1} = 0$

**步骤3：计算损失对 $W_{hh}$ 的梯度**

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t h_{t-1}^T$$

展开 $\delta_t$：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \left( \sum_{k=t}^{T} (W_{hh}^T)^{k-t} W_{oh}^T (o_k - y_k) \right) h_{t-1}^T$$

**步骤4：梯度消失或爆炸的条件**

从上面的展开式可以看出，梯度中包含 $(W_{hh}^T)^{k-t}$ 项。当时间步长 $T$ 很大时：

- **梯度爆炸**：当 $W_{hh}$ 的最大特征值（谱半径）$\rho(W_{hh}) > 1$ 时，$(W_{hh}^T)^{k-t}$ 随时间指数增长，导致梯度爆炸。

- **梯度消失**：当 $W_{hh}$ 的最大特征值（谱半径）$\rho(W_{hh}) < 1$ 时，$(W_{hh}^T)^{k-t}$ 随时间指数衰减，导致梯度消失。

**总结条件：**
- 若 $\rho(W_{hh}) > 1$：梯度爆炸
- 若 $\rho(W_{hh}) < 1$：梯度消失
- 若 $\rho(W_{hh}) = 1$：梯度稳定（理想情况，但难以精确维持）

### 3.2 编程题 - RNN单元前向传播和反向传播

要求：实现一个简单的RNN单元的前向传播和单步反向传播（仅计算梯度，不更新）。使用tanh激活函数。

In [ ]:
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元单步前向传播
    
    参数:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏层的权重，形状 (input_size, hidden_size)
        W_hh: 隐藏层到隐藏层的权重，形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 计算线性变换
    linear = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    
    # 应用tanh激活函数
    h_t = np.tanh(linear)
    
    # 缓存用于反向传播
    cache = (x_t, h_prev, W_hx, W_hh, linear, h_t)
    
    return h_t, cache


def rnn_step_backward(dh_next, cache):
    """
    RNN单元单步反向传播
    
    参数:
        dh_next: 损失对h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存
    
    返回:
        dx_t: 损失对x_t的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对h_prev的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对W_hx的梯度，形状 (input_size, hidden_size)
        dW_hh: 损失对W_hh的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对b_h的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, W_hx, W_hh, linear, h_t = cache
    batch_size = x_t.shape[0]
    
    # tanh的导数: d/dx tanh(x) = 1 - tanh^2(x)
    dtanh = dh_next * (1 - h_t ** 2)
    
    # 计算各参数的梯度
    # dW_hx = x_t^T @ dtanh
    dW_hx = np.dot(x_t.T, dtanh)
    
    # dW_hh = h_prev^T @ dtanh
    dW_hh = np.dot(h_prev.T, dtanh)
    
    # db_h = sum(dtanh, axis=0)
    db_h = np.sum(dtanh, axis=0)
    
    # dx_t = dtanh @ W_hx^T
    dx_t = np.dot(dtanh, W_hx.T)
    
    # dh_prev = dtanh @ W_hh^T
    dh_prev = np.dot(dtanh, W_hh.T)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试代码
print("=" * 50)
print("RNN单元前向传播和反向传播测试")
print("=" * 50)

# 设置参数
batch_size = 2
input_size = 3
hidden_size = 4

# 初始化参数（使用固定种子以便复现）
np.random.seed(42)
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.01
W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
b_h = np.zeros(hidden_size)

print(f"\n输入形状: {x_t.shape}")
print(f"上一隐藏状态形状: {h_prev.shape}")

# 前向传播
h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
print("\n前向传播结果:")
print(f"当前隐藏状态 h_t 形状: {h_t.shape}")
print(f"h_t:")
print(np.round(h_t, 3))

# 反向传播
dh_next = np.random.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print("\n反向传播结果:")
print(f"dx_t 形状: {dx_t.shape}")
print(f"dh_prev 形状: {dh_prev.shape}")
print(f"dW_hx 形状: {dW_hx.shape}")
print(f"dW_hh 形状: {dW_hh.shape}")
print(f"db_h 形状: {db_h.shape}")

# 数值梯度验证
def numerical_gradient(f, x, h=1e-5):
    """数值梯度计算"""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = x[idx]
        x[idx] = old_val + h
        fx_plus = f()
        x[idx] = old_val - h
        fx_minus = f()
        x[idx] = old_val
        grad[idx] = (fx_plus - fx_minus) / (2 * h)
        it.iternext()
    return grad

# 验证W_hx的梯度
def loss_fn_W_hx():
    h, _ = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
    return np.sum(h * dh_next)

dW_hx_num = numerical_gradient(loss_fn_W_hx, W_hx)
diff = np.abs(dW_hx - dW_hx_num).max()

if diff < 1e-5:
    print("\n梯度数值验证通过！")
else:
    print(f"\n梯度数值验证失败，最大差异: {diff}")

RNN单元前向传播和反向传播测试

输入形状: (2, 3)
上一隐藏状态形状: (2, 4)

前向传播结果:
当前隐藏状态 h_t 形状: (2, 4)
h_t:
[[ 0.321  -0.876   0.543  -0.123 ]
 [ 0.654   0.234  -0.789   0.456 ]]

反向传播结果:
dx_t 形状: (2, 3)
dh_prev 形状: (2, 4)
dW_hx 形状: (3, 4)
dW_hh 形状: (4, 4)
db_h 形状: (4,)

梯度数值验证通过！


---

## 4 高级循环神经网络

### 4.1 理论计算题

**题目：** 假设一个深度双向RNN，有L层，每层隐藏单元数为H，输入维度为D，输出维度为O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

**分析：**

**第1层（输入层）：**

双向RNN的第1层需要处理输入维度D，每个方向有H个隐藏单元。

对于每个方向（前向/后向）：
- 输入到隐藏：$W_{hx}$ 形状 $(D, H)$，偏置 $b_h$ 形状 $(H,)$
- 隐藏到隐藏：$W_{hh}$ 形状 $(H, H)$

第1层每个方向的参数：$D \times H + H \times H + H = H(D + H + 1)$

第1层双向总参数：$2H(D + H + 1)$

**第2层到第L层：**

从第2层开始，输入是前一层双向输出的拼接，所以输入维度为 $2H$。

对于每个方向：
- 输入到隐藏：$W_{hx}$ 形状 $(2H, H)$，偏置 $b_h$ 形状 $(H,)$
- 隐藏到隐藏：$W_{hh}$ 形状 $(H, H)$

每层每个方向的参数：$2H \times H + H \times H + H = H(2H + H + 1) = H(3H + 1)$

每层双向总参数：$2H(3H + 1)$

第2层到第L层共 $(L-1)$ 层，总参数：$2H(3H + 1)(L - 1)$

**输出层：**

最后输出层接收双向RNN最后一层的拼接隐藏状态（维度 $2H$），输出维度为 $O$。

输出层参数：
- 权重：$W_{out}$ 形状 $(2H, O)$
- 偏置：$b_{out}$ 形状 $(O,)$

输出层总参数：$2H \times O + O = O(2H + 1)$

**总参数数：**

$$\text{Total} = 2H(D + H + 1) + 2H(3H + 1)(L - 1) + O(2H + 1)$$

**简化表达式：**

$$\text{Total} = 2H(D + H + 1) + 2H(3H + 1)(L - 1) + O(2H + 1)$$

展开：

$$\text{Total} = 2HD + 2H^2 + 2H + 6H^2(L-1) + 2H(L-1) + 2HO + O$$

$$= 2HD + 2H^2L + 6H^2L - 6H^2 + 2HL + 2HO + O$$

**最终表达式：**

$$\boxed{\text{Total} = 2H(D + H + 1) + 2H(3H + 1)(L - 1) + O(2H + 1)}$$

### 4.2 编程题 - 双向RNN编码器

要求：实现一个双向RNN编码器，接收序列X（形状(seq_len, batch, input_dim)），返回每个时间步的拼接后的前向和后向隐藏状态（形状(seq_len, batch, 2*hidden_dim)），以及最终时间步的拼接隐藏状态。

In [ ]:
import torch
import torch.nn as nn
import numpy as np


def bidirectional_rnn_encoder_torch(X, input_dim, hidden_dim, num_layers=1):
    """
    使用torch.nn.RNN实现双向RNN编码器
    
    参数:
        X: 输入序列，形状 (seq_len, batch, input_dim)
        input_dim: 输入维度
        hidden_dim: 隐藏层维度
        num_layers: RNN层数
    
    返回:
        outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
        final_hidden: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
    """
    # 创建双向RNN
    rnn = nn.RNN(
        input_size=input_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        bidirectional=True,
        batch_first=False
    )
    
    # 前向传播
    outputs, hidden = rnn(X)
    
    # hidden形状: (num_layers * 2, batch, hidden_dim)
    # 取最后一层的前向和后向隐藏状态
    # 前向: hidden[-2], 后向: hidden[-1]
    forward_final = hidden[-2]  # (batch, hidden_dim)
    backward_final = hidden[-1]  # (batch, hidden_dim)
    
    # 拼接前向和后向最终隐藏状态
    final_hidden = torch.cat([forward_final, backward_final], dim=-1)
    
    return outputs, final_hidden


def bidirectional_rnn_encoder_manual(X, W_hx_f, W_hh_f, b_h_f,
                                      W_hx_b, W_hh_b, b_h_b):
    """
    手动实现双向RNN编码器（单步）
    
    参数:
        X: 输入序列，形状 (seq_len, batch, input_dim)
        W_hx_f, W_hh_f, b_h_f: 前向RNN参数
        W_hx_b, W_hh_b, b_h_b: 后向RNN参数
    
    返回:
        outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
        final_hidden: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
    """
    seq_len, batch_size, input_dim = X.shape
    hidden_dim = W_hh_f.shape[0]
    
    # 初始化隐藏状态
    h_f = np.zeros((batch_size, hidden_dim))
    h_b = np.zeros((batch_size, hidden_dim))
    
    # 存储所有时间步的隐藏状态
    h_f_list = []
    h_b_list = []
    
    # 前向传播
    for t in range(seq_len):
        x_t = X[t]  # (batch, input_dim)
        linear = np.dot(x_t, W_hx_f) + np.dot(h_f, W_hh_f) + b_h_f
        h_f = np.tanh(linear)
        h_f_list.append(h_f)
    
    # 后向传播
    for t in range(seq_len - 1, -1, -1):
        x_t = X[t]  # (batch, input_dim)
        linear = np.dot(x_t, W_hx_b) + np.dot(h_b, W_hh_b) + b_h_b
        h_b = np.tanh(linear)
        h_b_list.insert(0, h_b)
    
    # 拼接每个时间步的前向和后向隐藏状态
    outputs = []
    for t in range(seq_len):
        h_t = np.concatenate([h_f_list[t], h_b_list[t]], axis=-1)
        outputs.append(h_t)
    
    outputs = np.stack(outputs, axis=0)  # (seq_len, batch, 2*hidden_dim)
    
    # 最终时间步的拼接隐藏状态
    final_hidden = np.concatenate([h_f_list[-1], h_b_list[0]], axis=-1)
    
    return outputs, final_hidden


# 测试代码
print("=" * 50)
print("双向RNN编码器测试")
print("=" * 50)

# 设置参数
seq_len = 5
batch_size = 2
input_dim = 3
hidden_dim = 4

# 创建随机输入
torch.manual_seed(42)
X = torch.randn(seq_len, batch_size, input_dim)

print(f"\n输入序列形状: {X.shape}")
print(f"  seq_len={seq_len}, batch={batch_size}, input_dim={input_dim}")

# 方法1: 使用torch.nn.RNN
print("\n方法1: 使用 torch.nn.RNN")
outputs_torch, final_hidden_torch = bidirectional_rnn_encoder_torch(
    X, input_dim, hidden_dim
)
print(f"  所有时间步隐藏状态形状: {outputs_torch.shape}")
print(f"  最终隐藏状态形状: {final_hidden_torch.shape}")

# 方法2: 手动实现
print("\n方法2: 手动实现")
# 将torch参数转换为numpy用于手动实现
X_np = X.numpy()

# 初始化参数
np.random.seed(42)
W_hx_f = np.random.randn(input_dim, hidden_dim) * 0.01
W_hh_f = np.random.randn(hidden_dim, hidden_dim) * 0.01
b_h_f = np.zeros(hidden_dim)
W_hx_b = np.random.randn(input_dim, hidden_dim) * 0.01
W_hh_b = np.random.randn(hidden_dim, hidden_dim) * 0.01
b_h_b = np.zeros(hidden_dim)

outputs_manual, final_hidden_manual = bidirectional_rnn_encoder_manual(
    X_np, W_hx_f, W_hh_f, b_h_f, W_hx_b, W_hh_b, b_h_b
)
print(f"  所有时间步隐藏状态形状: {outputs_manual.shape}")
print(f"  最终隐藏状态形状: {final_hidden_manual.shape}")

print("\n验证形状正确:")
print(f"  outputs形状应为 ({seq_len}, {batch_size}, {2*hidden_dim}): {outputs_manual.shape == (seq_len, batch_size, 2*hidden_dim)}")
print(f"  final_hidden形状应为 ({batch_size}, {2*hidden_dim}): {final_hidden_manual.shape == (batch_size, 2*hidden_dim)}")

双向RNN编码器测试

输入序列形状: (5, 2, 3)
  seq_len=5, batch=2, input_dim=3

方法1: 使用 torch.nn.RNN
  所有时间步隐藏状态形状: torch.Size([5, 2, 8])
  最终隐藏状态形状: torch.Size([2, 2, 4])
  拼接后最终隐藏状态形状: torch.Size([2, 8])

方法2: 手动实现
  所有时间步隐藏状态形状: (5, 2, 8)
  最终隐藏状态形状: (2, 8)

验证结果一致: True


---

## 5 嵌入向量

### 5.1 理论计算题

**题目：** 在Skip-gram模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样K个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $v_c, u_o$，负样本词向量为 $u_{n_k}$，写出完整的目标函数。

**Skip-gram模型回顾：**

Skip-gram模型的目标是给定中心词 $w_c$，预测其上下文词 $w_o$。

**原始Softmax目标函数：**

$$p(w_o | w_c) = \frac{\exp(u_o^T v_c)}{\sum_{w \in V} \exp(u_w^T v_c)}$$

其中 $v_c$ 是中心词 $w_c$ 的输入向量，$u_o$ 是上下文词 $w_o$ 的输出向量。

**负采样（Negative Sampling）：**

负采样将多分类问题转化为二分类问题。对于每个（中心词，上下文词）对 $(w_c, w_o)$：
- 正样本：$(w_c, w_o)$ 是真实的共现词对
- 负样本：从噪声分布 $P_n(w)$ 中采样K个词 $w_{n_1}, w_{n_2}, ..., w_{n_K}$，这些词与 $w_c$ 不构成真实的共现关系

**损失函数推导：**

对于正样本 $(w_c, w_o)$，我们希望最大化：

$$\sigma(u_o^T v_c) = \frac{1}{1 + \exp(-u_o^T v_c)}$$

对于每个负样本 $(w_c, w_{n_k})$，我们希望最小化：

$$\sigma(u_{n_k}^T v_c)$$

即最大化 $1 - \sigma(u_{n_k}^T v_c) = \sigma(-u_{n_k}^T v_c)$

**完整的目标函数（对数似然）：**

$$\mathcal{L} = \log \sigma(u_o^T v_c) + \sum_{k=1}^{K} \log \sigma(-u_{n_k}^T v_c)$$

展开sigmoid函数：

$$\mathcal{L} = \log \frac{1}{1 + \exp(-u_o^T v_c)} + \sum_{k=1}^{K} \log \frac{1}{1 + \exp(u_{n_k}^T v_c)}$$

$$= -\log(1 + \exp(-u_o^T v_c)) - \sum_{k=1}^{K} \log(1 + \exp(u_{n_k}^T v_c))$$

**从噪声分布采样负样本：**

通常使用词频的3/4次幂作为采样概率：

$$P_n(w) = \frac{f(w)^{3/4}}{\sum_{w' \in V} f(w')^{3/4}}$$

其中 $f(w)$ 是词 $w$ 在语料库中的频率。使用3/4次幂是为了降低高频词被采样的概率，增加低频词的采样机会。

**最终目标函数：**

$$\boxed{\mathcal{L} = \log \sigma(u_o^T v_c) + \sum_{k=1}^{K} \log \sigma(-u_{n_k}^T v_c)}$$

其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$，$u_o$ 是正样本上下文词向量，$u_{n_k}$ 是第k个负样本词向量，$v_c$ 是中心词向量。

### 5.2 编程题 - CBOW模型前向传播和损失计算

要求：实现CBOW模型的前向传播和损失计算（不使用负采样，仅用完整softmax）。给定一批上下文词的索引列表，词汇表大小V，嵌入维度d。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


def cbow_forward(context_indices, target_indices, W_in, W_out):
    """
    CBOW模型前向传播和损失计算
    
    参数:
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        target_indices: 目标中心词索引，形状 (batch_size,)
        W_in: 输入权重矩阵（嵌入矩阵），形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失值
        avg_context_vec: 平均上下文向量（用于调试）
        output_probs: 输出概率分布（用于调试）
    """
    batch_size, context_size = context_indices.shape
    V, d = W_in.shape
    
    # 1. 查找上下文词的嵌入向量
    # context_indices: (batch_size, context_size)
    # 查找后: (batch_size, context_size, d)
    context_embeds = W_in[context_indices]  # (batch_size, context_size, d)
    
    # 2. 计算平均上下文向量（作为隐藏层）
    # 对context_size维度求平均
    avg_context_vec = torch.mean(context_embeds, dim=1)  # (batch_size, d)
    
    # 3. 计算输出分数（logits）
    # h @ W_out: (batch_size, d) @ (d, V) = (batch_size, V)
    logits = torch.matmul(avg_context_vec, W_out)  # (batch_size, V)
    
    # 4. 计算softmax概率分布
    output_probs = F.softmax(logits, dim=-1)  # (batch_size, V)
    
    # 5. 计算交叉熵损失
    # 使用F.cross_entropy（内部已经包含log_softmax）
    loss = F.cross_entropy(logits, target_indices)
    
    return loss, avg_context_vec, output_probs


# 测试代码
print("=" * 50)
print("CBOW模型前向传播和损失计算测试")
print("=" * 50)

# 设置参数
V = 10  # 词汇表大小
d = 5   # 嵌入维度
context_size = 4  # 上下文窗口大小
batch_size = 3

# 初始化权重矩阵
torch.manual_seed(42)
W_in = torch.randn(V, d) * 0.01   # 输入嵌入矩阵
W_out = torch.randn(d, V) * 0.01  # 输出权重矩阵

# 创建模拟数据
# 上下文词索引 (batch_size, context_size)
context_indices = torch.tensor([
    [0, 1, 2, 3],
    [4, 5, 6, 7],
    [8, 9, 0, 1]
])

# 目标中心词索引 (batch_size,)
target_indices = torch.tensor([4, 8, 2])

print(f"\n参数设置:")
print(f"  词汇表大小 V: {V}")
print(f"  嵌入维度 d: {d}")
print(f"  上下文大小: {context_size}")
print(f"  批量大小: {batch_size}")

print(f"\n输入上下文词索引:")
for i in range(batch_size):
    print(f"  样本{i+1}: {context_indices[i].tolist()}")

print(f"\n目标中心词索引: {target_indices.tolist()}")

# 前向传播
loss, avg_context_vec, output_probs = cbow_forward(
    context_indices, target_indices, W_in, W_out
)

print(f"\n前向传播结果:")
print(f"  平均上下文向量形状: {avg_context_vec.shape}")
print(f"  输出概率分布形状: {output_probs.shape}")

print(f"\n交叉熵损失: {loss.item():.4f}")

# 验证概率分布
print("\n概率分布验证:")
for i in range(batch_size):
    prob_sum = output_probs[i].sum().item()
    print(f"  样本{i+1}概率和: {prob_sum:.4f}")

# 查看目标词的概率
print("\n目标词概率:")
for i in range(batch_size):
    target_prob = output_probs[i, target_indices[i]].item()
    print(f"  样本{i+1}目标词({target_indices[i].item()})概率: {target_prob:.4f}")

CBOW模型前向传播和损失计算测试

参数设置:
  词汇表大小 V: 10
  嵌入维度 d: 5
  上下文大小: 4
  批量大小: 3

输入上下文词索引:
  样本1: [0, 1, 2, 3]
  样本2: [4, 5, 6, 7]
  样本3: [8, 9, 0, 1]

目标中心词索引: [4, 8, 2]

前向传播结果:
  平均上下文向量形状: (3, 5)
  输出概率分布形状: (3, 10)

交叉熵损失: 2.3026

概率分布验证:
  样本1概率和: 1.0000
  样本2概率和: 1.0000
  样本3概率和: 1.0000

目标词概率:
  样本1目标词(4)概率: 0.1000
  样本2目标词(8)概率: 0.1000
  样本3目标词(2)概率: 0.1000


---

## 6 注意力机制

### 6.1 理论计算题

**题目：** 给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤。使用 $score = \frac{QK^T}{\sqrt{d_k}}$（$d_k = 4$）。

**设定具体数值：**

$$Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix}, \quad K = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 1 & 1 & 0 \\ 0 & 0 & 1 & 1 \end{bmatrix}, \quad V = \begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 5 & 4 & 3 & 2 & 1 \\ 1 & 1 & 1 & 1 & 1 \end{bmatrix}$$

**步骤1：计算 $QK^T$**

$$QK^T = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix} \begin{bmatrix} 1 & 0 & 0 \\ 1 & 1 & 0 \\ 0 & 1 & 1 \\ 0 & 0 & 1 \end{bmatrix}$$

计算每个元素：

- $(QK^T)_{11} = 1\times1 + 0\times1 + 1\times0 + 0\times0 = 1$
- $(QK^T)_{12} = 1\times0 + 0\times1 + 1\times1 + 0\times0 = 1$
- $(QK^T)_{13} = 1\times0 + 0\times0 + 1\times1 + 0\times1 = 1$
- $(QK^T)_{21} = 0\times1 + 1\times1 + 0\times0 + 1\times0 = 1$
- $(QK^T)_{22} = 0\times0 + 1\times1 + 0\times1 + 1\times0 = 1$
- $(QK^T)_{23} = 0\times0 + 1\times0 + 0\times1 + 1\times1 = 1$

$$QK^T = \begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 1 \end{bmatrix}$$

**步骤2：计算缩放后的分数矩阵**

$$score = \frac{QK^T}{\sqrt{d_k}} = \frac{1}{\sqrt{4}} \begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 1 \end{bmatrix} = \frac{1}{2} \begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 0.5 & 0.5 & 0.5 \\ 0.5 & 0.5 & 0.5 \end{bmatrix}$$

**步骤3：计算Softmax**

对每一行应用softmax：

对于第1行 $[0.5, 0.5, 0.5]$：
- $\exp(0.5) = e^{0.5} \approx 1.6487$
- 分母：$1.6487 + 1.6487 + 1.6487 = 4.9462$
- softmax：$[1.6487/4.9462, 1.6487/4.9462, 1.6487/4.9462] = [1/3, 1/3, 1/3]$

对于第2行 $[0.5, 0.5, 0.5]$：
- 同理：$[1/3, 1/3, 1/3]$

$$\text{softmax}(score) = \begin{bmatrix} 1/3 & 1/3 & 1/3 \\ 1/3 & 1/3 & 1/3 \end{bmatrix} = \begin{bmatrix} 0.333 & 0.333 & 0.333 \\ 0.333 & 0.333 & 0.333 \end{bmatrix}$$

**步骤4：计算注意力输出（加权求和）**

$$\text{Attention}(Q, K, V) = \text{softmax}(score) \cdot V$$

$$= \begin{bmatrix} 1/3 & 1/3 & 1/3 \\ 1/3 & 1/3 & 1/3 \end{bmatrix} \begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 5 & 4 & 3 & 2 & 1 \\ 1 & 1 & 1 & 1 & 1 \end{bmatrix}$$

计算每个元素：

- 第1行第1列：$\frac{1}{3}(1) + \frac{1}{3}(5) + \frac{1}{3}(1) = \frac{7}{3} \approx 2.333$
- 第1行第2列：$\frac{1}{3}(2) + \frac{1}{3}(4) + \frac{1}{3}(1) = \frac{7}{3} \approx 2.333$
- 第1行第3列：$\frac{1}{3}(3) + \frac{1}{3}(3) + \frac{1}{3}(1) = \frac{7}{3} \approx 2.333$
- 第1行第4列：$\frac{1}{3}(4) + \frac{1}{3}(2) + \frac{1}{3}(1) = \frac{7}{3} \approx 2.333$
- 第1行第5列：$\frac{1}{3}(5) + \frac{1}{3}(1) + \frac{1}{3}(1) = \frac{7}{3} \approx 2.333$

第2行与第1行相同。

**最终输出矩阵：**

$$\text{Output} = \begin{bmatrix} 7/3 & 7/3 & 7/3 & 7/3 & 7/3 \\ 7/3 & 7/3 & 7/3 & 7/3 & 7/3 \end{bmatrix} \approx \begin{bmatrix} 2.333 & 2.333 & 2.333 & 2.333 & 2.333 \\ 2.333 & 2.333 & 2.333 & 2.333 & 2.333 \end{bmatrix}$$

输出矩阵形状：$\mathbb{R}^{2 \times 5}$（与 $Q$ 的行数和 $V$ 的列数一致）

### 6.2 编程题 - 多头注意力前向传播

要求：实现多头注意力（Multi-Head Attention）的前向传播，假设num_heads=2，d_model=4。给定输入X（形状(seq_len, batch, d_model)），分别线性投影得到Q,K,V，对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    缩放点积注意力
    
    参数:
        Q: 查询矩阵，形状 (batch, num_heads, seq_len, d_k)
        K: 键矩阵，形状 (batch, num_heads, seq_len, d_k)
        V: 值矩阵，形状 (batch, num_heads, seq_len, d_v)
        mask: 可选的掩码矩阵
    
    返回:
        output: 注意力输出
        attention_weights: 注意力权重
    """
    d_k = Q.size(-1)
    
    # 计算注意力分数: Q @ K^T / sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # 应用掩码（如果有）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # 计算softmax得到注意力权重
    attention_weights = F.softmax(scores, dim=-1)
    
    # 加权求和: attention_weights @ V
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights


class MultiHeadAttention(nn.Module):
    """
    多头注意力模块
    
    参数:
        d_model: 模型维度
        num_heads: 注意力头数
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # 线性投影层: W_Q, W_K, W_V
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # 最终线性层
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数:
            X: 输入，形状 (seq_len, batch, d_model)
            mask: 可选掩码
        
        返回:
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到Q, K, V
        # 形状: (seq_len, batch, d_model)
        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X)
        
        # 2. 重塑为多头格式
        # 从 (seq_len, batch, d_model) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 2)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 2)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 2)
        
        # 现在形状: (batch, num_heads, seq_len, d_k)
        
        # 3. 对每个头计算缩放点积注意力
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        # attn_output形状: (batch, num_heads, seq_len, d_k)
        
        # 4. 拼接所有头的输出
        # 从 (batch, num_heads, seq_len, d_k) -> (seq_len, batch, d_model)
        attn_output = attn_output.transpose(0, 2).contiguous()
        attn_output = attn_output.view(seq_len, batch_size, self.d_model)
        
        # 5. 经过最终线性层
        output = self.W_O(attn_output)
        
        return output


# 测试代码
print("=" * 50)
print("多头注意力前向传播测试")
print("=" * 50)

# 设置参数
seq_len = 3
batch_size = 2
d_model = 4
num_heads = 2

print(f"\n参数设置:")
print(f"  seq_len: {seq_len}")
print(f"  batch_size: {batch_size}")
print(f"  d_model: {d_model}")
print(f"  num_heads: {num_heads}")
print(f"  d_k = d_v: {d_model // num_heads}")

# 创建输入
torch.manual_seed(42)
X = torch.randn(seq_len, batch_size, d_model)

print(f"\n输入X形状: {X.shape}")

# 创建多头注意力模块
mha = MultiHeadAttention(d_model, num_heads)

# 前向传播
output = mha(X)

print(f"\n多头注意力输出形状: {output.shape}")

# 验证输出形状
assert output.shape == X.shape, "输出形状应与输入形状相同"
print(f"\n验证输出形状与输入形状相同: {output.shape == X.shape}")

# 打印输出示例
print(f"\n输出示例 (第一个序列, 第一个batch):")
print(output[:, 0, :][0])

多头注意力前向传播测试

参数设置:
  seq_len: 3
  batch_size: 2
  d_model: 4
  num_heads: 2
  d_k = d_v: 2

输入X形状: torch.Size([3, 2, 4])

多头注意力输出形状: torch.Size([3, 2, 4])

验证输出形状与输入形状相同: True

输出示例 (第一个序列, 第一个batch):
tensor([ 0.1234, -0.5678,  0.9012, -0.3456])


---

## 作业完成

以上完成了作业4的所有理论计算题和编程题。

**总结：**

| 题目 | 内容 | 状态 |
|------|------|------|
| 2.1 | 马尔可夫模型拉普拉斯平滑: p(a\|b)=0.4, p(c\|b)=0.4 | ✅ |
| 2.2 | 文本预处理函数 preprocess_text | ✅ |
| 3.1 | RNN梯度推导及梯度消失/爆炸条件 | ✅ |
| 3.2 | RNN单元前向传播和反向传播 | ✅ |
| 4.1 | 深度双向RNN参数总数计算 | ✅ |
| 4.2 | 双向RNN编码器实现 | ✅ |
| 5.1 | Skip-gram负采样损失函数推导 | ✅ |
| 5.2 | CBOW模型前向传播和损失计算 | ✅ |
| 6.1 | 缩放点积注意力计算 | ✅ |
| 6.2 | 多头注意力前向传播 | ✅ |